## ERA5 level data readin and integrate flux before averaging for rho calcs

### Read in ERA5 and calculate hourly integrated flux before resampling to monthly


In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import scipy
import sys 
import warnings
import matplotlib.pyplot as plt
from shapely.geometry import mapping
import cartopy.crs as ccrs
import cartopy.feature
warnings.filterwarnings('ignore')
import time as timer
start_all = timer.time()

In [2]:
dataf ="/Volumes/ESA_F4R/era/" 
datao ="/Volumes/ESA_F4R/ed_prepare/" 
datap ="/Users/ellendyer/Library/Mobile Documents/com~apple~CloudDocs/1SHARED_WORK/Work/3_ESA_GRANT/MODEL/plots/era/"

In [ ]:
#For selection and plotting
YR = 2010
time_bnds = (str(YR)+'-01-01',str(YR)+'-12-31')
lon_bnds, lat_bnds = (8, 32), (12,-15)
p_bnds = (30000,100000)

**Read in ERA5 data on pressure levels (hourly timesteps in fortnightly files)**
- *fortnightly files currently run from 1994-2024*
- resampled to monthly MS timestep
- shum multiplied by 1000 to convert from kg/kg --> g/kg
- pressure levels are divided by 100 to convert from Pa to hPa (only for fortnightly files)
- sort data by descending pressure levels (only for fortnightly files)


**Input file units:**
- plev - pa
- q - kg/kg
- u - m/s
- v - m/s

In [4]:
start = timer.time()
from functools import partial
def _preprocess_pres(x, lon_bnds, lat_bnds, p_bnds):
    return x.sel(lon=slice(*lon_bnds), lat=slice(*lat_bnds),
                 plev=slice(*p_bnds),drop=True)
partial_func_pres = partial(_preprocess_pres, lon_bnds=lon_bnds, lat_bnds=lat_bnds, p_bnds=p_bnds)

levs_M = {}
for M in np.arange(1,13):
    print(dataf+"era5/pressure_levels/era5_pressure_level_variables_central_africa_"+str(YR)+"-"+str("{:02d}".format(M))+"*.nc")
    #Reading in pressure level variables from ERA5
    ds_era_pres = xr.open_mfdataset(dataf+"era5/pressure_levels/era5_pressure_level_variables_central_africa_"+str(YR)+"-"+str("{:02d}".format(M))+"*.nc",
                                    drop_variables=['r','t','w'],
                                    preprocess=partial_func_pres,parallel=True).load()
                                    
    ds_era_pres = ds_era_pres.rename({'plev':'level','q':'Shum','u':'Uwnd','v':'Vwnd'})
    ds_era_pres['Shum'] = 1000.0*ds_era_pres['Shum']
    ds_era_pres['level'] = ds_era_pres['level']/100.0  
    ds_era_pres = ds_era_pres.sortby('level', ascending=False) 
    ds_era_pres = ds_era_pres.sortby('lat', ascending=True)
    #print(ds_era_pres)
    levs_M[M]=ds_era_pres
    ds_era_pres.close() 
end = timer.time()
length = end - start
print("ERA5 pressure level data read in took ", length, "seconds")

/Volumes/ESA_F4R/era/era5/pressure_levels/era5_pressure_level_variables_central_africa_2010-01*.nc
<xarray.Dataset> Size: 2GB
Dimensions:  (time: 744, level: 20, lat: 109, lon: 93)
Coordinates:
  * level    (level) float64 160B 1e+03 975.0 950.0 925.0 ... 400.0 350.0 300.0
  * lon      (lon) float32 372B 8.0 8.25 8.5 8.75 9.0 ... 30.25 30.5 30.75 31.0
  * lat      (lat) float32 436B -15.0 -14.75 -14.5 -14.25 ... 11.5 11.75 12.0
  * time     (time) datetime64[ns] 6kB 2010-01-01 ... 2010-01-31T23:00:00
Data variables:
    Shum     (time, level, lat, lon) float32 603MB 12.15 12.13 ... 0.1476 0.151
    Uwnd     (time, level, lat, lon) float32 603MB -3.108 -2.844 ... 10.67 10.19
    Vwnd     (time, level, lat, lon) float32 603MB 7.514 7.598 ... 0.8857 1.163
Attributes:
    Conventions:  CF-1.6
    history:      Sat Nov 15 09:32:15 2025: ncap2 -s plev=plev*100 era5_press...
    NCO:          netCDF Operators version 5.3.6 (Homepage = http://nco.sf.ne...
/Volumes/ESA_F4R/era/era5/pressure_lev

**Read in ERA5 surface pressure to calculate integrated moisture flux on hourly timestep**

In [5]:
start = timer.time()

from functools import partial
def _preprocess_land(x, lon_bnds, lat_bnds):
    x = x.sel(longitude=slice(*lon_bnds), latitude=slice(*lat_bnds),drop=True)
    return x
partial_func_land = partial(_preprocess_land, lon_bnds=lon_bnds, lat_bnds=lat_bnds)

surf_M = {}
for M in np.arange(1,13):
    #Reading in surface variables from ERA5 surface files
    ds_era_psfc = xr.open_mfdataset(dataf+"era5/era5_surface/era5_surface_pressure_central_africa_"+str(YR)+"-"+str("{:02d}".format(M))+".nc",
                                    drop_variables=['expver','number'],
                                    preprocess=partial_func_land,parallel=True).load()
    ds_era_psfc = ds_era_psfc.rename({'valid_time':'time','latitude':'lat',
                                      'longitude':'lon','sp':'Psfc'})
    Psfc = ds_era_psfc['Psfc']/100.0
    Psfc = Psfc.sortby('lat', ascending=True) 
    #print(Psfc.time)
    surf_M[M]=Psfc
    ds_era_psfc.close()
end = timer.time()
length = end - start
print("ERA5 surface data read in took ", length, "seconds")

ERA5 surface data read in took  7.915791749954224 seconds


**Write out one monthly pressure level dataset for recyling code called ds**
- calculate integrated moisture flux using surface pressure

In [6]:
import bulk_recycling_model.numerical_integration

for M in np.arange(1,13):
    # Integrate 10^-3 Shum Uwnd dp
    # Because the integration limits are from high pressure to low pressure, we need to invert the sign.
    integrand = -1 * 1e-3 * levs_M[M]["Shum"] * levs_M[M]["Uwnd"]
    levs_M[M]['Fx'] = bulk_recycling_model.numerical_integration.integrate_with_extrapolation(integrand, surf_M[M])
    # Units: mb x m/s
    
    # Integrate 10^-3 Shum Vwnd dp
    # Because the integration limits are from high pressure to low pressure, we need to invert the sign.
    integrand = -1 * 1e-3 * levs_M[M]["Shum"] * levs_M[M]["Vwnd"]
    levs_M[M]['Fy'] = bulk_recycling_model.numerical_integration.integrate_with_extrapolation(integrand, surf_M[M])
    # Units: mb x m/s
    print("done ",M)

done  1
done  2
done  3
done  4
done  5
done  6
done  7
done  8
done  9
done  10
done  11
done  12


**Write out one monthly pressure level dataset for recyling code called ds**
- resample to monthly timestep
- transpose dimensions so they run (lon,lat,level,time) as in recycling code
- save input ds to file

In [7]:
ds_list = []
for M in np.arange(1,13):
    ds_list.append(levs_M[M])
ds_era_pres_out = xr.concat(ds_list,dim='time')
print(ds_era_pres_out)

ds_era_pres_out = ds_era_pres_out.resample(time='MS').mean(dim='time')
ds_era_pres_out = ds_era_pres_out.transpose("lon", "lat", "level", "time",missing_dims='ignore')

<xarray.Dataset> Size: 23GB
Dimensions:  (time: 8760, level: 20, lat: 109, lon: 93)
Coordinates:
  * level    (level) float64 160B 1e+03 975.0 950.0 925.0 ... 400.0 350.0 300.0
  * lon      (lon) float32 372B 8.0 8.25 8.5 8.75 9.0 ... 30.25 30.5 30.75 31.0
  * lat      (lat) float32 436B -15.0 -14.75 -14.5 -14.25 ... 11.5 11.75 12.0
  * time     (time) datetime64[ns] 70kB 2010-01-01 ... 2010-12-31T23:00:00
Data variables:
    Shum     (time, level, lat, lon) float32 7GB 12.15 12.13 ... 0.05332 0.04897
    Uwnd     (time, level, lat, lon) float32 7GB -3.108 -2.844 ... 10.78 10.85
    Vwnd     (time, level, lat, lon) float32 7GB 7.514 7.598 ... -1.068 -1.353
    Fx       (time, lat, lon) float64 710MB -6.266 -5.971 ... -0.4998 -0.5731
    Fy       (time, lat, lon) float64 710MB 17.14 17.83 18.6 ... -3.882 -3.932
Attributes:
    Conventions:  CF-1.6
    history:      Sat Nov 15 09:32:15 2025: ncap2 -s plev=plev*100 era5_press...
    NCO:          netCDF Operators version 5.3.6 (Homepage =

In [9]:
ds_era_pres_out.to_netcdf(datao+"merge_erads_L_HI_"+str(YR)+".nc", mode='w', format='NETCDF4', engine='netcdf4')
end = timer.time()
length = end - start
print("Merging and dataset output took ", length, "seconds")

Merging and dataset output took  3058.55992603302 seconds
